# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name:Goh Yang Jie



**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

# Libraries

In [ ]:
## Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

# 1. Business Understanding
Goal: Predict medical insurance charges (in USD) for individuals based on personal and health-related attributes. This helps insurance companies accurately price premiums and assess financial risk for new policyholders, avoiding inaccurate pricing that impacts business profitability. This is a regression task as the target variable (expenses) is a continuous numerical value.

# 2. Data Understanding

## 2.1 Load dataset

In [ ]:
## Read *.csv file into pandas DataFrame
FILE_PATH = "insurance.csv"
df = pd.read_csv(FILE_PATH)
df

## 2.2 Summary Statistics

In [ ]:
## Understand the type of variable for each column
df.info()

In [ ]:
## Check for missing data
df.isna().sum()

In [ ]:
## Describe data distribution
df.describe(include='all')

## 2.3 Data Visualization

### 2.3.1 Understanding distribution of data

### 2.3.1.1 Understanding distribution of target

In [ ]:
## Understanding distribution of target
col_y = 'expenses'

df[col_y].hist()
plt.title('Distribution of Insurance Expenses')
plt.xlabel('expenses')
plt.ylabel('count')
plt.show()

# The target is right-skewed. A log-transform will be applied in data preparation
# to make the distribution more normal and improve model performance.

### 2.3.1.2 Understanding distribution of features

In [ ]:
## Understanding distribution of features
df.hist(figsize=(10, 8))
plt.suptitle('Feature Distributions')
plt.tight_layout()
plt.show()

In [ ]:
df.boxplot(rot=45, figsize=(8, 5))
plt.show()

### 2.3.2 Understanding relationship between variables

In [ ]:
## Understanding relationship between variables

# Correlation heatmap (numerical features only)
col_numeric = df.select_dtypes(include=['float64', 'int64']).columns
df_corr = df[col_numeric].corr()

sns.heatmap(df_corr, annot=True, cmap='YlGnBu')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Smoker status has a strong impact on expenses
sns.boxplot(x='smoker', y='expenses', data=df)
plt.title('Smoker vs Insurance Expenses')
plt.show()

In [ ]:
# Smokers with high BMI incur much higher charges - justifies bmi_smoker interaction feature
sns.scatterplot(x='bmi', y='expenses', hue='smoker', data=df)
plt.title('BMI vs Expenses by Smoker Status')
plt.show()

# 3. Data Preparation

## 3.1 Data Cleaning

In [ ]:
## Clean data

# Check and remove duplicates
print(f"Duplicates before cleaning: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Rows remaining after cleaning: {len(df)}")

# Encode categorical columns
df['sex'] = df['sex'].map({'male': 1, 'female': 0})
df['smoker'] = df['smoker'].map({'yes': 1, 'no': 0})
df = pd.get_dummies(df, columns=['region'], drop_first=True)

# Feature Engineering
# bmi_smoker: smokers with high BMI tend to have much higher charges.
# multiplying bmi by smoker flag captures this combined effect.
df['bmi_smoker'] = df['bmi'] * df['smoker']

# obese: costs increase significantly when BMI >= 30.
# binary flag to capture this threshold effect.
df['obese'] = (df['bmi'] >= 30).astype(int)

# log_expenses: expenses are right-skewed.
# Log-transforming makes the distribution more normal for better model training.
df['log_expenses'] = np.log(df['expenses'])

df.head()

## 3.2 Train-Test Split

In [ ]:
## Split data into train set and test set

# Separate features (X) and target (y)
col_y = 'log_expenses'
y = df[col_y]

# Drop original expenses and log_expenses from features
X = df.drop(['expenses', 'log_expenses'], axis=1)

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

# 4. Modelling

### 4.2 Train Model

In [ ]:
## Initialise and train model

# Initialize models
linr = LinearRegression()
rf   = RandomForestRegressor(n_estimators=500, random_state=42)
gbt  = GradientBoostingRegressor(random_state=42)

# Fit models
linr.fit(X_train, y_train)
rf.fit(X_train, y_train)
gbt.fit(X_train, y_train)

# Predict on test set
y_pred_linr = linr.predict(X_test)
y_pred_rf   = rf.predict(X_test)
y_pred_gbt  = gbt.predict(X_test)

# Evaluate all 3 baseline models
print("Baseline Model Comparison")
for name, y_pred in [("Linear Regression", y_pred_linr),
                      ("Random Forest",     y_pred_rf),
                      ("Gradient Boosting", y_pred_gbt)]:
    print(f"\n{name}")
    print("MAE: ",  mean_absolute_error(y_test, y_pred))
    print("RMSE:",  root_mean_squared_error(y_test, y_pred))
    print("R2:  ",  r2_score(y_test, y_pred))

# 5. Model Evaluation

In [ ]:
## Evaluate model

# Gradient Boosting scored the highest R2 - so we select it as the best model and evaluate it in detail
y_pred = gbt.predict(X_test)
print("MAE: ",  mean_absolute_error(y_test, y_pred))
print("MSE: ",  mean_squared_error(y_test, y_pred))
print("RMSE:",  root_mean_squared_error(y_test, y_pred))
print("R2:  ",  r2_score(y_test, y_pred))

# R2 tells us how well the model explains the variation in insurance charges
# RMSE is in log scale since we trained on log_expenses, lower means better
# Both metrics are suitable for regression problems

In [ ]:
## New data
# Testing with a 35 year old male smoker with high BMI from Southeast region
new_data = pd.DataFrame({
    'age': [35], 'sex': [1], 'bmi': [32.5], 'children': [2], 'smoker': [1],
    'region_northwest': [0], 'region_southeast': [1], 'region_southwest': [0],
    'bmi_smoker': [32.5], 'obese': [1]
})
new_data = new_data.reindex(columns=X_train.columns, fill_value=0)

## Predict
#np.exp reverses the log transform to get the actual charge in USD
log_pred = gbt.predict(new_data)
print(f"Predicted Insurance Charges: ${np.exp(log_pred[0]):,.2f}")

## Iterative model development


In [ ]:
## Further feature engineering / feature selection